# Test SafeRoute — 3 chế độ trên dataset tiếng Việt (`dynabench_vn.json`)

Dataset tự tạo (458 mẫu: chủ quyền VN, bảo mật nội bộ, lăng mạ...). Schema:
`uid`, `label`, `rules` (list `{index, category_id, policy}`), `transcript` (dạng `'User':`/`'Agent':`).

Pipeline: ghép `rules` → khối policy đánh số → **format prompt chuẩn DynaGuard non-CoT** →
chạy **3 chế độ**: (1) Small 1.7B, (2) Large 8B, (3) **SafeRoute** ([router](https://huggingface.co/StevenMup2004/SafeRoute-final), ngưỡng 0.6) + Oracle (trần).


In [ ]:
# 1. Cài + clone DynaGuard (đúng prompt/hằng số gốc)
!pip -q install -U transformers accelerate huggingface_hub scikit-learn
import os, sys
if not os.path.exists('DynaGuard'):
    !git clone --depth 1 https://github.com/montehoover/DynaGuard.git
sys.path.append('DynaGuard')
from constants import (DYNAGUARD_SYSTEM_PROMPT, DYNAGUARD_CONTENT_TEMPLATE,
                       LABEL_OPENING, DYNAGUARD_USER_TAG, DYNAGUARD_AGENT_TAG,
                       POS_LABEL, NEG_LABEL)  # POS=FAIL, NEG=PASS
print('POS(FAIL)/NEG(PASS) =', repr(POS_LABEL), repr(NEG_LABEL))

## 2. Cấu hình & nạp dataset VN

In [ ]:
# 2. Config + load
SMALL_MODEL = "tomg-group-umd/DynaGuard-1.7B"
LARGE_MODEL = "tomg-group-umd/DynaGuard-8B"
ROUTER_REPO = "StevenMup2004/SafeRoute-final"
THRESHOLD   = 0.6
LIMIT       = 0          # 0 = tất cả
SMALL_BS    = 16
LARGE_BS    = 8

import os, json
DATA_PATH = "dynabench_vn.json"
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Hãy chọn dynabench_vn.json ...")
        up = files.upload(); DATA_PATH = list(up.keys())[0]
    except Exception as e:
        raise FileNotFoundError(f"Không thấy {DATA_PATH}. ({e})")

raw = json.load(open(DATA_PATH, encoding="utf-8"))
if LIMIT: raw = raw[:LIMIT]

def build_policy(rec):
    # Ghép list rules -> khối đánh số cho <rules>
    return "\n".join(f"{r['index']}. {r['policy']}" for r in rec["rules"])

examples = [{
    "uid":        rec["uid"],
    "gold":       rec["label"],
    "policy":     build_policy(rec),
    "transcript": rec["transcript"],
} for rec in raw]

from collections import Counter
print(f"{len(examples)} mẫu | nhãn:", dict(Counter(e["gold"] for e in examples)))
print("--- policy mẫu[0] ---"); print(examples[0]["policy"][:300])
print("--- transcript mẫu[0] ---"); print(examples[0]["transcript"][:300])

## 3. Hàm format prompt chuẩn DynaGuard (non-CoT)

In [ ]:
# 3. Prompt non-CoT
def format_user_agent_tags(t, u=DYNAGUARD_USER_TAG, a=DYNAGUARD_AGENT_TAG):
    t = t.replace("User:", u).replace("Agent:", a)
    t = t.replace("'User':", u).replace("'Agent:'", a)
    return t

def build_prompt(tokenizer, policy, transcript):
    transcript = format_user_agent_tags(transcript)
    user = DYNAGUARD_CONTENT_TEMPLATE.format(policy=policy, conversation=transcript)
    msgs = [
        {"role": "system",    "content": DYNAGUARD_SYSTEM_PROMPT},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": f"{LABEL_OPENING}\n"},
    ]
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, continue_final_message=True, enable_thinking=False)

print(build_prompt.__name__, "OK")

## 4. Pass A — 1.7B: trích feature 2048 + dự đoán PASS/FAIL (so logit)

In [ ]:
# 4. Pass A
import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

def load_lm(name):
    tok = AutoTokenizer.from_pretrained(name); tok.padding_side = "left"
    if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
    m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="auto").eval()
    return tok, m

def label_ids(tok):
    return (tok.encode(NEG_LABEL, add_special_tokens=False)[0],   # PASS
            tok.encode(POS_LABEL, add_special_tokens=False)[0])   # FAIL

@torch.no_grad()
def run_pass(name, batch_size, want_feature):
    tok, model = load_lm(name)
    pass_id, fail_id = label_ids(tok)
    prompts = [build_prompt(tok, e["policy"], e["transcript"]) for e in examples]
    preds, feats = [], [] if want_feature else None
    for i in tqdm(range(0, len(prompts), batch_size), desc=name.split("/")[-1]):
        enc = tok(prompts[i:i+batch_size], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        out = model(**enc, output_hidden_states=want_feature)
        last = out.logits[:, -1, :]
        is_fail = (last[:, fail_id] > last[:, pass_id]).tolist()
        preds += [POS_LABEL if f else NEG_LABEL for f in is_fail]
        if want_feature:
            feats.append(out.hidden_states[-1][:, -1, :].float().cpu().numpy())
    if want_feature: feats = np.concatenate(feats, 0)
    del model; torch.cuda.empty_cache()
    return preds, feats

preds_small, feats = run_pass(SMALL_MODEL, SMALL_BS, want_feature=True)
print("feature:", feats.shape, "| preds_small[:5]:", preds_small[:5])

## 5. Pass B — 8B: dự đoán PASS/FAIL

In [ ]:
# 5. Pass B
preds_large, _ = run_pass(LARGE_MODEL, LARGE_BS, want_feature=False)
print("preds_large[:5]:", preds_large[:5])

## 6. Nạp router & quyết định định tuyến

In [ ]:
# 6. Router
import torch.nn as nn
from huggingface_hub import hf_hub_download

class RouterMLP(nn.Module):
    def __init__(self, input_dim=2048):
        super().__init__()
        self.cls = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(1024, 512),       nn.BatchNorm1d(512),  nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 256),        nn.BatchNorm1d(256),  nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
    def forward(self, x): return self.cls(x).squeeze(-1)

device = "cuda" if torch.cuda.is_available() else "cpu"
router = RouterMLP(2048).to(device)
ckpt = torch.load(hf_hub_download(ROUTER_REPO, "model.pt"), map_location=device)
state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
router.load_state_dict(state, strict=False); router.eval()

with torch.no_grad():
    probs = torch.sigmoid(router(torch.from_numpy(feats).float().to(device))).cpu().numpy()
route_to_large = probs > THRESHOLD
print(f"%route sang 8B: {route_to_large.mean()*100:.1f}% (ngưỡng {THRESHOLD})")

## 7. 3 chế độ + Oracle

In [ ]:
# 7. Đánh giá
from sklearn.metrics import f1_score, accuracy_score, classification_report

gold = [e["gold"] for e in examples]
pred_small  = preds_small
pred_large  = preds_large
pred_route  = [pl if b else ps for ps, pl, b in zip(preds_small, preds_large, route_to_large)]
pred_oracle = [g if (ps == g or pl == g) else ps
               for ps, pl, g in zip(preds_small, preds_large, gold)]

def report(name, pred, usage=None):
    acc = accuracy_score(gold, pred)
    f1f = f1_score(gold, pred, pos_label=POS_LABEL)
    f1m = f1_score(gold, pred, average="macro")
    u = "" if usage is None else f" | %dùng 8B: {usage*100:5.1f}"
    print(f"{name:12s}  acc={acc:.4f}  F1(FAIL)={f1f:.4f}  macroF1={f1m:.4f}{u}")
    return dict(mode=name, acc=acc, f1_fail=f1f, macro_f1=f1m,
                large_usage=(None if usage is None else float(usage)))

print(f"=== 3 CHẾ ĐỘ trên dynabench_vn.json ({len(examples)} mẫu) ===")
results = [
    report("1) Small",     pred_small,  0.0),
    report("2) Large",     pred_large,  1.0),
    report("3) SafeRoute", pred_route,  float(route_to_large.mean())),
    report("(*) Oracle",   pred_oracle, None),
]
t_true = [1 if (ps != g and pl == g) else 0 for ps, pl, g in zip(preds_small, preds_large, gold)]
t_pred = [int(b) for b in route_to_large]
print(f"\nRouting F1: {f1_score(t_true, t_pred):.4f} | #mẫu-khó thực: {sum(t_true)} | #route-8B: {sum(t_pred)}")
print("\nChi tiết SafeRoute:")
print(classification_report(gold, pred_route, labels=[POS_LABEL, NEG_LABEL], digits=4))

## 8. Lưu kết quả

In [ ]:
# 8. Lưu JSON
OUT = "saferoute_vn_test.json"
json.dump({
    "data": DATA_PATH, "threshold": THRESHOLD,
    "small_model": SMALL_MODEL, "large_model": LARGE_MODEL, "router": ROUTER_REPO,
    "num": len(examples), "large_usage_saferoute": float(route_to_large.mean()),
    "modes": results,
    "per_example": [
        {"uid": e["uid"], "gold": g, "pred_small": ps, "pred_large": pl,
         "route_to_large": bool(b), "router_prob": float(p),
         "pred_saferoute": (pl if b else ps)}
        for e, g, ps, pl, b, p in zip(examples, gold, preds_small, preds_large, route_to_large, probs)
    ],
}, open(OUT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Đã lưu", OUT)
try:
    from google.colab import files; files.download(OUT)
except Exception: pass